# Lean-RGC v54 miniF2F Experiment Colab

This notebook sets up Lean-RGC, fetches the Lean 4 miniF2F benchmark, generates Lean-RGC task JSONL, and runs a kernel-RPC smoke experiment with bivariate contextual quotient, face taxonomy, and obstruction tower artifacts.

Primary pass gates:

- `n_failures == 0` in the main audit
- `stateful_kernel_rpc_audit == true`
- contextual audit uses `kernel_context_cache == true`
- `source_check_calls == 0`
- `baseline_missing == 0`
- `row_degenerate == false` and `column_degenerate == false`
- nonempty `dual_face_taxonomy` and `tower_faces`

Start with `MINIF2F_LIMIT=16`. Scale only after the smoke run passes.


In [ ]:
%%bash
set -euo pipefail
REPO_URL="${REPO_URL:-https://github.com/abhorrence-of-Gods/lean-rgc-automation-stack-v51.git}"
REPO_DIR="${REPO_DIR:-/content/lean-rgc-automation-stack-v51}"
if [ ! -d "$REPO_DIR/.git" ]; then
  git clone "$REPO_URL" "$REPO_DIR"
else
  cd "$REPO_DIR"
  git pull --ff-only
fi
cat > /content/lean_rgc_env.sh <<EOF
export REPO_URL="$REPO_URL"
export REPO_DIR="$REPO_DIR"
EOF
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
git rev-parse --short HEAD
git status --short


In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
python -m pip install -U pip
python -m pip install -e . pytest
python -m lean_rgc.cli minif2f-fetch --help | head -n 40
python -m lean_rgc.cli obstruction-tower --help | head -n 40


In [ ]:
%%bash
set -euo pipefail
if [ ! -f "$HOME/.elan/env" ]; then
  curl https://raw.githubusercontent.com/leanprover/elan/master/elan-init.sh -sSf | sh -s -- -y --default-toolchain stable
fi
source "$HOME/.elan/env"
lean --version
lake --version


## Fetch and Build miniF2F

The benchmark checkout is kept under `benchmarks/miniF2F`. It is not vendored into Lean-RGC. The build can take time on a fresh Colab runtime because miniF2F imports Mathlib and FormalConjectures.


In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
source "$HOME/.elan/env"
cd "$REPO_DIR"

lean-rgc minif2f-fetch \
  --out benchmarks/miniF2F \
  --summary-out runs/minif2f_fetch_report.json

cd benchmarks/miniF2F
TOOLCHAIN="$(cat lean-toolchain)"
elan toolchain install "$TOOLCHAIN"
lake update
lake exe cache get || true
lake build MiniF2F


In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
cd "$REPO_DIR"

export MINIF2F_LIMIT="${MINIF2F_LIMIT:-16}"
mkdir -p examples runs
lean-rgc minif2f-tasks \
  --repo benchmarks/miniF2F \
  --split valid \
  --limit "$MINIF2F_LIMIT" \
  --out "examples/minif2f_valid${MINIF2F_LIMIT}_tasks.jsonl" \
  --summary-out "examples/minif2f_valid${MINIF2F_LIMIT}_report.json"

python - <<'PY'
import json
from pathlib import Path
import os
limit = os.environ.get("MINIF2F_LIMIT", "16")
report = Path(f"examples/minif2f_valid{limit}_report.json")
print(report.read_text())
rows = [json.loads(x) for x in Path(f"examples/minif2f_valid{limit}_tasks.jsonl").read_text().splitlines() if x.strip()]
print("first task:")
print(json.dumps(rows[0], ensure_ascii=False, indent=2)[:1800])
PY


## Create a Stable Action File

This avoids relying on any local example action file beyond a fresh clone. The set is intentionally conservative for the first smoke run.


In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
mkdir -p examples
cat > examples/core_tactics_rows_minif2f_v54.jsonl <<'EOF'
{"action_id":"trivial","tactic":"trivial","tactic_class":"exact","cost_estimate":1.0}
{"action_id":"rfl","tactic":"rfl","tactic_class":"exact","cost_estimate":1.0}
{"action_id":"exact_rfl","tactic":"exact rfl","tactic_class":"exact","cost_estimate":1.0}
{"action_id":"assumption","tactic":"assumption","tactic_class":"exact","cost_estimate":1.0}
{"action_id":"simp","tactic":"simp","tactic_class":"simp","cost_estimate":1.5}
{"action_id":"simp_all","tactic":"simp_all","tactic_class":"simp","cost_estimate":2.0}
{"action_id":"constructor","tactic":"constructor","tactic_class":"constructor","cost_estimate":1.0}
{"action_id":"constructor_assumption","tactic":"constructor <;> assumption","tactic_class":"constructor","cost_estimate":1.5}
{"action_id":"intro","tactic":"intro","tactic_class":"intro","cost_estimate":1.0}
{"action_id":"intros","tactic":"intros","tactic_class":"intro","cost_estimate":1.0}
{"action_id":"intro_simp","tactic":"intro x; simp","tactic_class":"intro_simp","cost_estimate":2.0}
{"action_id":"intro_assumption","tactic":"intro h; assumption","tactic_class":"intro_exact","cost_estimate":1.5}
{"action_id":"intro_rfl","tactic":"intro x; rfl","tactic_class":"intro_exact","cost_estimate":1.5}
{"action_id":"constructor_simp","tactic":"constructor <;> simp","tactic_class":"constructor","cost_estimate":2.0}
{"action_id":"constructor_rfl","tactic":"constructor <;> rfl","tactic_class":"constructor","cost_estimate":1.5}
{"action_id":"norm_num","tactic":"norm_num","tactic_class":"arith","cost_estimate":2.0}
{"action_id":"decide","tactic":"decide","tactic_class":"decidable","cost_estimate":2.0}
EOF
wc -l examples/core_tactics_rows_minif2f_v54.jsonl
head -n 5 examples/core_tactics_rows_minif2f_v54.jsonl


## Kernel RPC Smoke Probe

This verifies that the native Lean worker can load the miniF2F project. If this cell passes, the pipeline should use `--workdir benchmarks/miniF2F` and `--import-mode preserve`.


In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
source "$HOME/.elan/env"
cd "$REPO_DIR"

python -m lean_rgc.native_worker \
  --lean-cmd "lake env lean" \
  --exec-mode kernel_rpc \
  --workdir benchmarks/miniF2F \
  --print-command

CMD="$(python -m lean_rgc.native_worker --lean-cmd 'lake env lean' --exec-mode kernel_rpc --workdir benchmarks/miniF2F --print-command)"
printf '%s\n' \
  '{"id":"load","cmd":"load_project","imports":["MiniF2F.ProblemImports"]}' \
  '{"id":"stop","cmd":"shutdown"}' \
  | bash -lc "$CMD"


## Run miniF2F Kernel-RPC Smoke Experiment

This is the first real run. Keep it small first. The output directory is stored in `/tmp/lean_rgc_last_run.txt` for later summary cells.


In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
source "$HOME/.elan/env"
cd "$REPO_DIR"

export MINIF2F_LIMIT="${MINIF2F_LIMIT:-16}"
RUN="runs/minif2f_valid${MINIF2F_LIMIT}_v54_kernel_tower_$(date +%Y%m%d_%H%M%S)"
mkdir -p runs

lean-rgc pipeline \
  --tasks "examples/minif2f_valid${MINIF2F_LIMIT}_tasks.jsonl" \
  --actions examples/core_tactics_rows_minif2f_v54.jsonl \
  --out "$RUN" \
  --audit-mode server \
  --server-backend native \
  --native-exec-mode kernel_rpc \
  --server-no-fallback \
  --lean-cmd "lake env lean" \
  --workdir benchmarks/miniF2F \
  --timeout-s 60 \
  --import-mode preserve \
  --max-actions 64 \
  --premise-contextual-bivariate \
  --premise-contextual-quotient \
  --premise-contextual-max-premises 12 \
  --premise-contextual-max-left 3 \
  --premise-contextual-max-right 5 \
  --bivariate-audit-budget 128 \
  --premise-contextual-baseline-required \
  --skip-vacuous-premise-quotient \
  --repair-face-ledger \
  --face-taxonomy \
  --obstruction-tower \
  2>&1 | tee "$RUN.log"

echo "$RUN" > /tmp/lean_rgc_last_run.txt
echo "RUN=$RUN"


## Summarize the Run

This cell reports proof/audit performance and whether the bivariate/taxonomy/tower layers produced nondegenerate artifacts.


In [ ]:
from pathlib import Path
import json

repo = Path('/content/lean-rgc-automation-stack-v51')
run_file = Path('/tmp/lean_rgc_last_run.txt')
if not run_file.exists():
    raise SystemExit('No /tmp/lean_rgc_last_run.txt found. Run the experiment cell first.')
run = Path(run_file.read_text().strip())
if not run.is_absolute():
    run = repo / run
print('RUN =', run)

def load_json(path):
    path = Path(path)
    if not path.exists():
        return None
    return json.loads(path.read_text())

def count_jsonl(path):
    path = Path(path)
    if not path.exists():
        return None
    return sum(1 for line in path.read_text().splitlines() if line.strip())

def pick(d, keys):
    if not isinstance(d, dict):
        return {k: None for k in keys}
    return {k: d.get(k) for k in keys}

main = load_json(run/'audit/server_summary.json')
ctx = load_json(run/'premise_contextual_audit/server_summary.json')
fp = load_json(run/'premise_contextual/premise_contextual_fingerprint_report.json')
quot = load_json(run/'premise_contextual/premise_quotient_report.json')
tax = load_json(run/'premise_contextual/face_taxonomy/dual_face_taxonomy_report.json')
tower = load_json(run/'premise_contextual/obstruction_tower/tower_summary.json')

keys = ['n_failures','n_responses','statuses','success_rate','timeout_rate','stateful_kernel_rpc_audit','kernel_context_cache','context_cache_hits','source_check_calls']
for name, data in [('main', main), ('ctx', ctx)]:
    print('\n==', name)
    print(json.dumps(pick(data, keys), indent=2, ensure_ascii=False))

print('\n== bivariate')
print(json.dumps(pick(fp, ['n_fingerprints','n_unique_premise_use_ids','n_unique_context_pairs','baseline_missing','row_degenerate','column_degenerate']), indent=2))
print('\n== quotient')
print(json.dumps(pick(quot, ['n_premise_use_rows','n_classes','quotient_status']), indent=2))
print('\n== taxonomy')
print(json.dumps(pick(tax, ['n_rows','n_concepts','n_taxonomy_faces','n_retrieval_allowed_faces','dual_source_counts','retrieval_blocker_counts']), indent=2))
print('\n== tower')
print(json.dumps(pick(tower, ['n_objects','n_transcripts','n_faces','n_dual_components','n_boundaries','n_promotions','n_next_actions','n_retrieval_candidates','next_action_counts','next_reason_counts']), indent=2))

print('\nline counts:')
for rel in [
    'audit/responses.jsonl',
    'premise_contextual/premise_use_rows.jsonl',
    'premise_contextual/premise_contextual_fingerprints.jsonl',
    'premise_contextual/premise_quotient_classes.jsonl',
    'premise_contextual/face_taxonomy/dual_face_taxonomy.jsonl',
    'premise_contextual/obstruction_tower/tower_faces.jsonl',
    'premise_contextual/obstruction_tower/tower_next_actions.jsonl',
    'premise_contextual/obstruction_tower/tower_retrieval_candidates.jsonl',
]:
    print(rel, count_jsonl(run/rel))


## Inspect Tower Next Actions

These are not tactics yet. They are structured feedback: generate more rows, generate separator contexts, split carrier-mixed faces, resolve lower boundaries, or block retrieval.


In [ ]:
from pathlib import Path
import json
from collections import Counter

repo = Path('/content/lean-rgc-automation-stack-v51')
run = Path(Path('/tmp/lean_rgc_last_run.txt').read_text().strip())
if not run.is_absolute():
    run = repo / run
path = run/'premise_contextual/obstruction_tower/tower_next_actions.jsonl'
rows = [json.loads(x) for x in path.read_text().splitlines() if x.strip()] if path.exists() else []
print('n_next_actions =', len(rows))
print('action_kind:', Counter(r.get('action_kind') for r in rows))
print('reason:', Counter(r.get('reason') for r in rows))
for row in rows[:20]:
    print(json.dumps(row, ensure_ascii=False, indent=2)[:1200])


## Optional Scale-Up

Run this only after the smoke gates pass. Increase `MINIF2F_LIMIT`, `--max-actions`, and `--bivariate-audit-budget` gradually. miniF2F/Mathlib runs are CPU and memory heavy on Colab.


In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
source "$HOME/.elan/env"
cd "$REPO_DIR"

MINIF2F_LIMIT=64
lean-rgc minif2f-tasks \
  --repo benchmarks/miniF2F \
  --split valid \
  --limit "$MINIF2F_LIMIT" \
  --out "examples/minif2f_valid${MINIF2F_LIMIT}_tasks.jsonl" \
  --summary-out "examples/minif2f_valid${MINIF2F_LIMIT}_report.json"

RUN="runs/minif2f_valid${MINIF2F_LIMIT}_v54_kernel_tower_$(date +%Y%m%d_%H%M%S)"
mkdir -p runs
lean-rgc pipeline \
  --tasks "examples/minif2f_valid${MINIF2F_LIMIT}_tasks.jsonl" \
  --actions examples/core_tactics_rows_minif2f_v54.jsonl \
  --out "$RUN" \
  --audit-mode server \
  --server-backend native \
  --native-exec-mode kernel_rpc \
  --server-no-fallback \
  --lean-cmd "lake env lean" \
  --workdir benchmarks/miniF2F \
  --timeout-s 90 \
  --import-mode preserve \
  --max-actions 128 \
  --premise-contextual-bivariate \
  --premise-contextual-quotient \
  --premise-contextual-max-premises 24 \
  --premise-contextual-max-left 4 \
  --premise-contextual-max-right 6 \
  --bivariate-audit-budget 512 \
  --premise-contextual-baseline-required \
  --skip-vacuous-premise-quotient \
  --repair-face-ledger \
  --face-taxonomy \
  --obstruction-tower \
  2>&1 | tee "$RUN.log"

echo "$RUN" > /tmp/lean_rgc_last_run.txt
echo "RUN=$RUN"


## Download Results Archive

This packs the latest run into `/content` so it can be downloaded from the Colab file browser.


In [ ]:
%%bash
set -euo pipefail
source /content/lean_rgc_env.sh
cd "$REPO_DIR"
RUN="$(cat /tmp/lean_rgc_last_run.txt)"
BASENAME="$(basename "$RUN")"
zip -qr "/content/${BASENAME}.zip" "$RUN" "$RUN.log" || zip -qr "/content/${BASENAME}.zip" "$RUN"
echo "/content/${BASENAME}.zip"
